# 16 — GroupBy Deep Dive

`groupby()` is the heart of analytical Pandas. This notebook covers split-apply-combine,
every aggregation pattern, transform vs agg vs apply, and production-level idioms.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 1000

employees = pd.DataFrame({
    'emp_id':     range(1001, 1001 + n),
    'name':       [f'Employee_{i:04d}' for i in range(n)],
    'department': np.random.choice(['Engineering', 'Sales', 'HR', 'Marketing', 'Finance'], n),
    'level':      np.random.choice(['Junior', 'Mid', 'Senior', 'Lead'], n),
    'salary':     np.random.randint(40000, 180000, n).astype(float),
    'bonus':      np.random.randint(0, 30000, n).astype(float),
    'experience': np.random.randint(1, 20, n),
    'city':       np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Chennai'], n),
    'rating':     np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0], n, p=[0.05, 0.1, 0.25, 0.4, 0.2]),
    'join_year':  np.random.choice(range(2015, 2024), n),
})

print(f"Shape: {employees.shape}")
employees.head(3)

## 1. The Split-Apply-Combine Pattern

1. **Split** the DataFrame into groups
2. **Apply** a function to each group independently
3. **Combine** the results

In [ ]:
# Basic groupby
grp = employees.groupby('department')
print(f"Type: {type(grp)}")
print(f"Number of groups: {grp.ngroups}")
print(f"Group sizes:")
print(grp.size())

In [ ]:
# Mean salary per department
print(grp['salary'].mean().round(0))

## 2. agg() — Multiple Aggregations

In [ ]:
# Named aggregations (Pandas >= 0.25) — cleanest syntax
dept_stats = employees.groupby('department').agg(
    headcount      = ('emp_id',  'count'),
    avg_salary     = ('salary',  'mean'),
    median_salary  = ('salary',  'median'),
    min_salary     = ('salary',  'min'),
    max_salary     = ('salary',  'max'),
    avg_experience = ('experience', 'mean'),
    total_bonus    = ('bonus',   'sum'),
).round(0)

print(dept_stats)

In [ ]:
# Multiple functions per column (dict syntax)
result = employees.groupby('department').agg({
    'salary': ['mean', 'median', 'std'],
    'bonus':  ['sum', 'mean'],
    'rating': 'mean'
}).round(1)

# Flatten multi-level column names
result.columns = ['_'.join(c).strip() for c in result.columns]
print(result)

In [ ]:
# Custom aggregation function in agg()
def salary_range(x):
    return x.max() - x.min()

result = employees.groupby('department')['salary'].agg(
    avg='mean',
    std='std',
    salary_range=salary_range,
    percentile_90=lambda x: x.quantile(0.9)
).round(0)
print(result)

## 3. Multi-Key GroupBy

In [ ]:
# Group by multiple columns
dept_level_stats = employees.groupby(['department', 'level']).agg(
    count      = ('emp_id', 'count'),
    avg_salary = ('salary', 'mean')
).round(0)

print(dept_level_stats.head(12))

In [ ]:
# Unstacking — pivot the inner index to columns
level_salary = (
    employees
    .groupby(['department', 'level'])['salary']
    .mean()
    .round(0)
    .unstack('level')
)
print(level_salary)

## 4. transform() — Group-Level Values Broadcast to Row Level

In [ ]:
# transform() returns same shape as input — aligns with original index
# Use case 1: add department avg salary as a column (for comparison)
employees['dept_avg_salary'] = employees.groupby('department')['salary'].transform('mean')

# Use case 2: salary relative to department median (z-score within group)
employees['salary_vs_dept'] = employees['salary'] - employees['dept_avg_salary']

print(employees[['name', 'department', 'salary', 'dept_avg_salary', 'salary_vs_dept']].head(8))

In [ ]:
# Use case 3: normalize salary within department (group z-score)
employees['salary_zscore'] = employees.groupby('department')['salary'].transform(
    lambda x: (x - x.mean()) / x.std()
).round(2)

print("Group-normalized salary:")
print(employees[['department', 'salary', 'salary_zscore']].head(6))

In [ ]:
# Use case 4: fill missing salary with department median (from notebook 11)
df_with_na = employees.copy()
df_with_na.loc[np.random.choice(n, 50, replace=False), 'salary'] = np.nan

df_with_na['salary'] = df_with_na.groupby('department')['salary'].transform(
    lambda x: x.fillna(x.median())
)
print(f"NaN salary after transform fill: {df_with_na['salary'].isna().sum()}")

## 5. apply() — Most Flexible but Slowest

In [ ]:
# apply() receives each group as a DataFrame/Series
# Use when agg() and transform() can't express your logic

# Get top-3 earners per department
top3_per_dept = (
    employees
    .groupby('department', group_keys=False)
    .apply(lambda g: g.nlargest(3, 'salary'))
    [['name', 'department', 'salary', 'level']]
)
print(top3_per_dept.head(12))

In [ ]:
# apply returning a scalar (same as agg)
salary_iqr = employees.groupby('department')['salary'].apply(
    lambda x: x.quantile(0.75) - x.quantile(0.25)
).round(0)
print("Salary IQR by department:")
print(salary_iqr)

## 6. agg vs transform vs apply — When to Use Which

| Method | Output Shape | Use Case |
|--------|-------------|----------|
| `agg()` | **Reduced** (one row per group) | Summary statistics |
| `transform()` | **Same shape** as input | Add group-level feature to each row |
| `apply()` | **Flexible** | Complex group-level logic |

In [ ]:
g = employees.groupby('department')['salary']

agg_result       = g.agg('mean')       # 5 rows (one per dept)
transform_result = g.transform('mean') # 1000 rows (same as df)

print(f"agg result shape:       {agg_result.shape}")
print(f"transform result shape: {transform_result.shape}")

## 7. filter() — Drop Entire Groups

In [ ]:
# Keep only departments with > 190 employees
large_depts = employees.groupby('department').filter(lambda g: len(g) > 190)
print(f"Original: {len(employees)}  After filter: {len(large_depts)}")
print(large_depts['department'].value_counts())

In [ ]:
# Keep only departments where avg salary > 100k
high_salary_depts = employees.groupby('department').filter(
    lambda g: g['salary'].mean() > 100000
)
print(high_salary_depts['department'].value_counts())

## 8. GroupBy with observed= for Categoricals

In [ ]:
# Without observed=True, all category combinations appear (even if empty)
employees['department'] = employees['department'].astype('category')

# observed=True is CRITICAL for categorical groupby in Pandas 2.2+
result = employees.groupby('department', observed=True)['salary'].mean().round(0)
print(result)

## 9. GroupBy on DatetimeIndex

In [ ]:
# Generate time-series sales data
np.random.seed(42)
sales_ts = pd.DataFrame({
    'date':     pd.date_range('2022-01-01', periods=365, freq='D'),
    'revenue':  np.random.randint(50000, 500000, 365),
    'category': np.random.choice(['Electronics', 'Clothing', 'Food'], 365)
})

# GroupBy on datetime-derived column
sales_ts['month'] = sales_ts['date'].dt.month
monthly = sales_ts.groupby('month')['revenue'].agg(['sum', 'mean']).round(0)
print(monthly)

In [ ]:
# GroupBy on Grouper (for DatetimeIndex)
ts_indexed = sales_ts.set_index('date')

weekly_by_cat = ts_indexed.groupby([
    pd.Grouper(freq='W'),
    'category'
])['revenue'].sum().unstack('category').fillna(0)

print(weekly_by_cat.head())

## 10. GroupBy Iteration — Accessing Individual Groups

In [ ]:
# Iterate over groups (rare — usually vectorized operations are better)
for dept_name, dept_df in employees.groupby('department', observed=True):
    top_earner = dept_df.nlargest(1, 'salary')[['name', 'salary']].iloc[0]
    print(f"{dept_name:15} — top earner: {top_earner['name']} (₹{top_earner['salary']:,.0f})")

## 11. GroupBy Cumulative Operations

In [ ]:
# Running totals and cumulative stats within groups
employees_sorted = employees.sort_values(['department', 'salary'])

employees_sorted['cumsum_salary_in_dept'] = (
    employees_sorted.groupby('department', observed=True)['salary'].cumsum()
)
employees_sorted['cummax_salary_in_dept'] = (
    employees_sorted.groupby('department', observed=True)['salary'].cummax()
)
employees_sorted['rank_in_dept'] = (
    employees_sorted.groupby('department', observed=True)['salary']
    .rank(ascending=False, method='dense').astype(int)
)

print(employees_sorted[
    ['department', 'name', 'salary', 'rank_in_dept']
].head(10))

## 12. Real-World — Headcount + Payroll Dashboard

In [ ]:
dashboard = employees.groupby(['department', 'level'], observed=True).agg(
    headcount    = ('emp_id', 'count'),
    avg_salary   = ('salary', 'mean'),
    total_payroll= ('salary', 'sum'),
    avg_rating   = ('rating', 'mean'),
    avg_exp      = ('experience', 'mean'),
).round(0)

# Add payroll share within department
dashboard['dept_payroll_share'] = (
    dashboard['total_payroll'] /
    dashboard.groupby('department', level=0)['total_payroll'].transform('sum') * 100
).round(1)

print(dashboard)